In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('/home/arshad/Network-project/data/cleaned_cicids2017.csv')
print(f"✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"✅ Labels: {df['Label'].nunique()} unique attack types")

/tmp/ipykernel_8966/2475852050.py:6: DtypeWarning: Columns (80,81,82) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/arshad/Network-project/data/cleaned_cicids2017.csv')


✅ Loaded: 2,574,264 rows × 83 columns
✅ Labels: 15 unique attack types


In [2]:
# These features are most meaningful for network security analysis
# Selected based on what's observable and relevant to each attack type
selected_features = [
    # Flow basics
    'Destination Port', 'Flow Duration',
    
    # Packet counts
    'Total Fwd Packets', 'Total Backward Packets',
    
    # Packet sizes
    'Fwd Packet Length Max', 'Fwd Packet Length Mean',
    'Bwd Packet Length Max', 'Bwd Packet Length Mean',
    'Packet Length Mean', 'Packet Length Std',
    
    # Flow rates
    'Flow Bytes/s', 'Flow Packets/s',
    'Fwd Packets/s', 'Bwd Packets/s',
    
    # Inter-arrival times
    'Flow IAT Mean', 'Flow IAT Std',
    'Fwd IAT Mean', 'Bwd IAT Mean',
    
    # TCP Flags (critical for attack detection)
    'SYN Flag Count', 'ACK Flag Count',
    'FIN Flag Count', 'RST Flag Count',
    'PSH Flag Count', 'URG Flag Count',
    
    # Window & segment info
    'Init_Win_bytes_forward', 'Init_Win_bytes_backward',
    'min_seg_size_forward',
    
    # Activity
    'Active Mean', 'Idle Mean',
    'Down/Up Ratio', 'Average Packet Size',

    # Labels
    'Label', 'MITRE_Tactic', 'MITRE_Technique', 'MITRE_Tech_Name',
    'source_file'
]

df_selected = df[selected_features].copy()
print(f"✅ Selected {len(selected_features) - 5} features + label columns")
print(f"📊 Shape: {df_selected.shape}")

✅ Selected 31 features + label columns
📊 Shape: (2574264, 36)


In [3]:
# This helps us write better alert text templates per attack
feature_cols = [col for col in selected_features 
                if col not in ['Label','MITRE_Tactic','MITRE_Technique',
                               'MITRE_Tech_Name','source_file']]

# Mean value of each feature per attack type
attack_profile = df_selected.groupby('Label')[feature_cols].mean().round(2)

print("📊 Attack Behavior Profiles (mean values per attack):")
print(attack_profile[['Flow Duration', 'SYN Flag Count', 
                       'Flow Packets/s', 'Packet Length Mean',
                       'Flow Bytes/s']].to_string())

📊 Attack Behavior Profiles (mean values per attack):
                            Flow Duration  SYN Flag Count  Flow Packets/s  Packet Length Mean  Flow Bytes/s
Label                                                                                                      
BENIGN                       1.185850e+07            0.06        53448.57              114.14    1691701.57
Bot                          3.532787e+05            0.00        43882.61               51.56     289396.10
DDoS                         1.695689e+07            0.00          155.39              736.94      60509.01
DoS GoldenEye                2.312105e+07            0.00            8.85              501.50        688.71
DoS Hulk                     7.630960e+07            0.00         1723.55              791.09      37923.55
DoS Slowhttptest             6.066875e+07            0.13         1579.85               83.05    1324834.75
DoS slowloris                6.083195e+07            0.27         1375.75          

In [4]:
def generate_alert_text(row):
    """
    Converts a network flow row into a natural language alert description.
    This text will be fed into the embedding model for MITRE ATT&CK mapping.
    """
    
    # Protocol inference from flags
    syn  = int(row.get('SYN Flag Count', 0))
    ack  = int(row.get('ACK Flag Count', 0))
    fin  = int(row.get('FIN Flag Count', 0))
    rst  = int(row.get('RST Flag Count', 0))
    psh  = int(row.get('PSH Flag Count', 0))
    urg  = int(row.get('URG Flag Count', 0))
    
    flags = []
    if syn: flags.append('SYN')
    if ack: flags.append('ACK')
    if fin: flags.append('FIN')
    if rst: flags.append('RST')
    if psh: flags.append('PSH')
    if urg: flags.append('URG')
    flag_str = ', '.join(flags) if flags else 'no TCP flags'

    # Flow characteristics
    duration    = float(row.get('Flow Duration', 0)) / 1e6  # microseconds to seconds
    pkt_rate    = float(row.get('Flow Packets/s', 0))
    byte_rate   = float(row.get('Flow Bytes/s', 0))
    pkt_len     = float(row.get('Packet Length Mean', 0))
    fwd_pkts    = int(row.get('Total Fwd Packets', 0))
    bwd_pkts    = int(row.get('Total Backward Packets', 0))
    dst_port    = int(row.get('Destination Port', 0))
    iat_mean    = float(row.get('Flow IAT Mean', 0))
    down_up     = float(row.get('Down/Up Ratio', 0))
    avg_pkt     = float(row.get('Average Packet Size', 0))
    idle_mean   = float(row.get('Idle Mean', 0))

    # Port context
    port_context = {
        80: 'HTTP web traffic',   443: 'HTTPS web traffic',
        22: 'SSH service',        21: 'FTP service',
        23: 'Telnet service',     3389: 'RDP service',
        53: 'DNS service',        3306: 'MySQL database',
        8080: 'HTTP alternate',   445: 'SMB file sharing'
    }
    port_desc = port_context.get(dst_port, f'port {dst_port}')

    # Flow direction balance
    if fwd_pkts + bwd_pkts > 0:
        fwd_ratio = fwd_pkts / (fwd_pkts + bwd_pkts)
        if fwd_ratio > 0.8:
            direction = "predominantly unidirectional (mostly forward)"
        elif fwd_ratio < 0.2:
            direction = "predominantly unidirectional (mostly backward)"
        else:
            direction = "bidirectional"
    else:
        direction = "unknown direction"

    # Intensity descriptors
    rate_desc  = "very high" if pkt_rate > 1000 else \
                 "high"      if pkt_rate > 100  else \
                 "moderate"  if pkt_rate > 10   else "low"

    size_desc  = "large"     if pkt_len > 500  else \
                 "medium"    if pkt_len > 100   else "small"

    dur_desc   = "long"      if duration > 60  else \
                 "medium"    if duration > 5    else "short"

    # Build the alert text
    alert = (
        f"Network flow targeting {port_desc} with {flag_str} flags. "
        f"Flow is {direction} with {fwd_pkts} forward and {bwd_pkts} backward packets. "
        f"Packet rate is {rate_desc} at {pkt_rate:.1f} packets per second "
        f"with {size_desc} average packet size of {pkt_len:.1f} bytes. "
        f"Flow duration is {dur_desc} at {duration:.2f} seconds "
        f"with byte rate of {byte_rate:.1f} bytes per second. "
        f"Mean inter-arrival time is {iat_mean:.1f} microseconds. "
        f"Average packet size is {avg_pkt:.1f} bytes "
        f"with down/up ratio of {down_up:.2f}. "
        f"Idle mean is {idle_mean:.1f} microseconds."
    )
    
    return alert

print("✅ Alert text generator defined!")

# Test it on one row
sample = df_selected.iloc[5000]
print("\n📝 Sample Alert Text:")
print("-" * 60)
print(generate_alert_text(sample))
print("-" * 60)
print(f"Label: {sample['Label']}")
print(f"MITRE: {sample['MITRE_Technique']} — {sample['MITRE_Tech_Name']}")

✅ Alert text generator defined!

📝 Sample Alert Text:
------------------------------------------------------------
Network flow targeting DNS service with no TCP flags flags. Flow is bidirectional with 1 forward and 1 backward packets. Packet rate is moderate at 42.2 packets per second with small average packet size of 71.3 bytes. Flow duration is short at 0.05 seconds with byte rate of 3436.1 bytes per second. Mean inter-arrival time is 47437.0 microseconds. Average packet size is 107.0 bytes with down/up ratio of 1.00. Idle mean is 0.0 microseconds.
------------------------------------------------------------
Label: BENIGN
MITRE: nan — nan


In [5]:
from tqdm import tqdm
tqdm.pandas()

# Work on a stratified sample — balanced across attack types
# This is smart: we don't need all 2.5M rows for the embedding phase
print("📊 Sampling dataset (stratified by attack type)...")

# For BENIGN take 10,000 samples, for attacks take all
benign_sample   = df_selected[df_selected['Label'] == 'BENIGN'].sample(10000, random_state=42)
attack_sample   = df_selected[df_selected['Label'] != 'BENIGN']

df_sample = pd.concat([benign_sample, attack_sample], ignore_index=True)
df_sample = df_sample.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(f"✅ Sample size: {len(df_sample):,} rows")
print(f"\n📊 Sample Label Distribution:")
print(df_sample['Label'].value_counts())

📊 Sampling dataset (stratified by attack type)...
✅ Sample size: 435,878 rows

📊 Sample Label Distribution:
Label
DoS Hulk                      172849
DDoS                          128016
PortScan                       90819
DoS GoldenEye                  10286
BENIGN                         10000
FTP-Patator                     5933
DoS slowloris                   5385
DoS Slowhttptest                5228
SSH-Patator                     3219
Bot                             1953
Web Attack - Brute Force        1470
Web Attack - XSS                 652
Infiltration                      36
Web Attack - SQL Injection        21
Heartbleed                        11
Name: count, dtype: int64


In [6]:
print("⏳ Generating alert texts...")
df_sample['alert_text'] = df_sample.progress_apply(generate_alert_text, axis=1)

print(f"\n✅ Alert texts generated!")
print(f"\n📝 Example alerts by attack type:")
print("=" * 70)

for label in df_sample['Label'].unique()[:5]:
    row = df_sample[df_sample['Label'] == label].iloc[0]
    print(f"\n🏷️  {label}")
    print(f"   {row['alert_text'][:200]}...")

⏳ Generating alert texts...


100%|████████████████████████████████| 435878/435878 [00:15<00:00, 28459.07it/s]



✅ Alert texts generated!

📝 Example alerts by attack type:

🏷️  DoS GoldenEye
   Network flow targeting HTTP web traffic with ACK flags. Flow is predominantly unidirectional (mostly forward) with 2 forward and 0 backward packets. Packet rate is low at 0.3 packets per second with s...

🏷️  PortScan
   Network flow targeting port 2718 with PSH flags. Flow is bidirectional with 1 forward and 1 backward packets. Packet rate is very high at 24691.4 packets per second with small average packet size of 3...

🏷️  DoS Hulk
   Network flow targeting HTTP web traffic with FIN flags. Flow is bidirectional with 6 forward and 7 backward packets. Packet rate is low at 0.2 packets per second with large average packet size of 854....

🏷️  DDoS
   Network flow targeting HTTP web traffic with PSH flags. Flow is bidirectional with 3 forward and 4 backward packets. Packet rate is high at 201.9 packets per second with large average packet size of 1...

🏷️  BENIGN
   Network flow targeting HTTP web traffic 

In [7]:
SAMPLE_PATH = '/home/arshad/Network-project/data/sample_with_alerts.csv'
df_sample.to_csv(SAMPLE_PATH, index=False)

print(f"✅ Saved sample with alert texts!")
print(f"📊 Shape: {df_sample.shape}")
print(f"📁 Location: {SAMPLE_PATH}")

✅ Saved sample with alert texts!
📊 Shape: (435878, 37)
📁 Location: /home/arshad/Network-project/data/sample_with_alerts.csv


In [8]:
# Fix NaN in MITRE columns for BENIGN rows
df_sample['MITRE_Tactic']    = df_sample['MITRE_Tactic'].fillna('None')
df_sample['MITRE_Technique'] = df_sample['MITRE_Technique'].fillna('None')
df_sample['MITRE_Tech_Name'] = df_sample['MITRE_Tech_Name'].fillna('None')

# Verify
print("✅ MITRE columns fixed!")
print(df_sample[['Label','MITRE_Tactic','MITRE_Technique']]
      .drop_duplicates()
      .sort_values('Label')
      .to_string())

✅ MITRE columns fixed!
                            Label MITRE_Tactic MITRE_Technique
13                         BENIGN         None            None
156                           Bot       TA0011           T1071
5                            DDoS       TA0040           T1498
0                   DoS GoldenEye       TA0040           T1499
2                        DoS Hulk       TA0040           T1499
16               DoS Slowhttptest       TA0040           T1499
47                  DoS slowloris       TA0040           T1499
50                    FTP-Patator       TA0006           T1110
1977                   Heartbleed       TA0001           T1210
18504                Infiltration       TA0008           T1570
1                        PortScan       TA0007           T1046
41                    SSH-Patator       TA0006           T1110
215      Web Attack - Brute Force       TA0001           T1190
29029  Web Attack - SQL Injection       TA0001           T1190
1396             Web Attack - XS

In [9]:
SAMPLE_PATH = '/home/arshad/Network-project/data/sample_with_alerts.csv'
df_sample.to_csv(SAMPLE_PATH, index=False)
print(f"✅ Re-saved with fixes!")
print(f"📊 Final shape: {df_sample.shape}")

✅ Re-saved with fixes!
📊 Final shape: (435878, 37)
